# ***`Libraries`***

In [50]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models


# ***`Read Data`***

In [51]:
df = pd.read_excel('Preprocessed Dataset.xlsx')
df

,Followers,Total Revenue,Years Joined,Rating Quality,Good Review,Med Review,Bad Review
0,0.060135,0.008283,1.806134,0.453583,0.631159,1.618767,1.080557
1,-0.071209,-0.047449,-1.814746,0.037897,-0.497662,1.196371,0.308262
2,-0.009052,0.008283,0.599174,0.424516,0.641790,-0.172147,1.292087
3,-0.046442,0.008283,0.599174,-0.066678,-0.096943,0.975558,1.761924
4,-0.072807,-0.053651,-0.607786,0.144730,0.344560,-0.269972,0.509950
...,...,...,...,...,...,...,...
1794,-0.072008,-0.054545,-0.004306,0.165504,-0.858078,-0.724797,-0.707270
1795,-0.075683,-0.054932,-1.814746,-2.227588,-1.622587,-0.724797,-0.707270
1796,-0.075683,-0.054932,-1.211266,-3.055399,-2.303363,-0.724797,-0.707270
1797,-0.071129,-0.052300,-1.814746,-0.478144,-0.076643,-0.724797,-0.707270


# ***`K-Bin Discretizer`***

In [52]:
from sklearn.preprocessing import KBinsDiscretizer

# 1. Xác định các cột số
numeric_cols = df.select_dtypes(include='number').columns.tolist()

# 2. Chuẩn bị labels và KBinsDiscretizer
labels = ["Low", "Medium", "High"]
disc = KBinsDiscretizer(
    n_bins=3,
    encode='ordinal',
    strategy='kmeans'
)

# 3. Fit và transform chỉ trên các cột số
binned = disc.fit_transform(df[numeric_cols]).astype(int)

# 4. Chuyển thành DataFrame, map sang nhãn rồi gán vào df mới
df_binned = df.copy()
df_binned[numeric_cols] = pd.DataFrame(binned, columns=numeric_cols).applymap(lambda x: labels[x])

# 5. Đổi tên tất cả cột thành chữ thường
df_binned = df_binned.rename(columns={col: col.lower() for col in df_binned.columns})

# Kết quả
df_binned

<ipython-input-52-698de6633408>:19: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_binned[numeric_cols] = pd.DataFrame(binned, columns=numeric_cols).applymap(lambda x: labels[x])


,followers,total revenue,years joined,rating quality,good review,med review,bad review
0,Medium,High,High,High,High,High,High
1,Low,Low,Low,High,Medium,High,Medium
2,Medium,High,High,High,High,Low,High
3,Low,High,High,High,Medium,High,High
4,Low,Low,Medium,High,High,Low,Medium
...,...,...,...,...,...,...,...
1794,Low,Low,Medium,High,Medium,Low,Low
1795,Low,Low,Low,Low,Low,Low,Low
1796,Low,Low,Low,Low,Low,Low,Low
1797,Low,Low,Low,Medium,Medium,Low,Low


# ***`One-hot encoding`***

In [53]:
df_onehot = pd.get_dummies(df_binned)
df_onehot

,followers_High,followers_Low,followers_Medium,total revenue_High,total revenue_Low,total revenue_Medium,years joined_High,years joined_Low,years joined_Medium,rating quality_High,...,rating quality_Medium,good review_High,good review_Low,good review_Medium,med review_High,med review_Low,med review_Medium,bad review_High,bad review_Low,bad review_Medium
0,False,False,True,True,False,False,True,False,False,True,...,False,True,False,False,True,False,False,True,False,False
1,False,True,False,False,True,False,False,True,False,True,...,False,False,False,True,True,False,False,False,False,True
2,False,False,True,True,False,False,True,False,False,True,...,False,True,False,False,False,True,False,True,False,False
3,False,True,False,True,False,False,True,False,False,True,...,False,False,False,True,True,False,False,True,False,False
4,False,True,False,False,True,False,False,False,True,True,...,False,True,False,False,False,True,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1794,False,True,False,False,True,False,False,False,True,True,...,False,False,False,True,False,True,False,False,True,False
1795,False,True,False,False,True,False,False,True,False,False,...,False,False,True,False,False,True,False,False,True,False
1796,False,True,False,False,True,False,False,True,False,False,...,False,False,True,False,False,True,False,False,True,False
1797,False,True,False,False,True,False,False,True,False,False,...,True,False,False,True,False,True,False,False,True,False


In [54]:
df_binned

,followers,total revenue,years joined,rating quality,good review,med review,bad review
0,Medium,High,High,High,High,High,High
1,Low,Low,Low,High,Medium,High,Medium
2,Medium,High,High,High,High,Low,High
3,Low,High,High,High,Medium,High,High
4,Low,Low,Medium,High,High,Low,Medium
...,...,...,...,...,...,...,...
1794,Low,Low,Medium,High,Medium,Low,Low
1795,Low,Low,Low,Low,Low,Low,Low
1796,Low,Low,Low,Low,Low,Low,Low
1797,Low,Low,Low,Medium,Medium,Low,Low


# ***`FP-Max`***

In [55]:
from mlxtend.frequent_patterns import fpmax, fpgrowth, apriori

# Tìm các tập mục phổ biến tối đại
frequent_itemsets = fpmax(df_onehot, min_support=0.2, use_colnames=True)
frequent_itemsets = pd.DataFrame(frequent_itemsets)
frequent_itemsets.head(2)

,support,itemsets
0,0.252918,"(rating quality_High, good review_High, med re..."
1,0.200111,"(bad review_Medium, followers_Low)"


In [56]:
# 1
df_binned['RatingHigh_GoodHigh_BadMedium'] = (
    (df_binned['rating quality'] == 'High') &
    (df_binned['good review'] == 'High') &
    (df_binned['bad review'] == 'Medium')
).astype(int)

# 2
df_binned['RevenueLow_YearsMedium_FollowersLow'] = (
    (df_binned['total revenue'] == 'Low') &
    (df_binned['years joined'] == 'Medium') &
    (df_binned['followers'] == 'Low')
).astype(int)

# 3
df_binned['RatingHigh_GoodHigh_MedMedium'] = (
    (df_binned['rating quality'] == 'High') &
    (df_binned['good review'] == 'High') &
    (df_binned['med review'] == 'Medium')
).astype(int)

# 4
df_binned['RatingHigh_YearsMedium_GoodHigh'] = (
    (df_binned['rating quality'] == 'High') &
    (df_binned['years joined'] == 'Medium') &
    (df_binned['good review'] == 'High')
).astype(int)

# 5
df_binned['RatingHigh_BadLow_FollowersLow_RevenueLow_GoodHigh'] = (
    (df_binned['rating quality'] == 'High') &
    (df_binned['bad review'] == 'Low') &
    (df_binned['followers'] == 'Low') &
    (df_binned['total revenue'] == 'Low') &
    (df_binned['good review'] == 'High')
).astype(int)

# 6
df_binned['RatingHigh_YearsMedium_RevenueLow'] = (
    (df_binned['rating quality'] == 'High') &
    (df_binned['years joined'] == 'Medium') &
    (df_binned['total revenue'] == 'Low')
).astype(int)

# 7
df_binned['RatingHigh_MedLow_FollowersLow_RevenueLow_GoodHigh'] = (
    (df_binned['rating quality'] == 'High') &
    (df_binned['med review'] == 'Low') &
    (df_binned['followers'] == 'Low') &
    (df_binned['total revenue'] == 'Low') &
    (df_binned['good review'] == 'High')
).astype(int)

# 8
df_binned['RatingHigh_BadLow_MedLow_FollowersLow_RevenueLow'] = (
    (df_binned['rating quality'] == 'High') &
    (df_binned['bad review'] == 'Low') &
    (df_binned['med review'] == 'Low') &
    (df_binned['followers'] == 'Low') &
    (df_binned['total revenue'] == 'Low')
).astype(int)

# 9
df_binned['RevenueLow_YearsMedium_MedLow'] = (
    (df_binned['total revenue'] == 'Low') &
    (df_binned['years joined'] == 'Medium') &
    (df_binned['med review'] == 'Low')
).astype(int)

# 10
df_binned['RatingHigh_YearsMedium_FollowersLow'] = (
    (df_binned['rating quality'] == 'High') &
    (df_binned['years joined'] == 'Medium') &
    (df_binned['followers'] == 'Low')
).astype(int)

# 11
df_binned['RatingHigh_BadMedium_RevenueLow'] = (
    (df_binned['rating quality'] == 'High') &
    (df_binned['bad review'] == 'Medium') &
    (df_binned['total revenue'] == 'Low')
).astype(int)

# 12
df_binned['RatingHigh_FollowersLow_RevenueLow_GoodHigh_YearsHigh'] = (
    (df_binned['rating quality'] == 'High') &
    (df_binned['followers'] == 'Low') &
    (df_binned['total revenue'] == 'Low') &
    (df_binned['good review'] == 'High') &
    (df_binned['years joined'] == 'High')
).astype(int)

# 13
df_binned['RatingHigh_BadLow_MedLow_RevenueLow_GoodHigh'] = (
    (df_binned['rating quality'] == 'High') &
    (df_binned['bad review'] == 'Low') &
    (df_binned['med review'] == 'Low') &
    (df_binned['total revenue'] == 'Low') &
    (df_binned['good review'] == 'High')
).astype(int)

# 14
df_binned['YearsMedium_MedLow_FollowersLow'] = (
    (df_binned['years joined'] == 'Medium') &
    (df_binned['med review'] == 'Low') &
    (df_binned['followers'] == 'Low')
).astype(int)

# 15
df_binned['RevenueLow_YearsMedium_BadLow'] = (
    (df_binned['total revenue'] == 'Low') &
    (df_binned['years joined'] == 'Medium') &
    (df_binned['bad review'] == 'Low')
).astype(int)

# 16
df_binned['BadMedium_FollowersLow'] = (
    (df_binned['bad review'] == 'Medium') &
    (df_binned['followers'] == 'Low')
).astype(int)

In [58]:
df_binned.to_excel('Binary Features.xlsx', index=False)